# Run experiments on DKT using ASSISTments 2009

## Imports

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import time
import random
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/education-ml-research"

DATA_DIR = os.path.join(BASE_DIR, "ASSISTments2009")
RESULTS_DIR = os.path.join(DATA_DIR, "reliability_experiment_irt")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory exists:", os.path.isdir(BASE_DIR))
print("Dataset directory exists:", os.path.isdir(DATA_DIR))
print("Results directory:", RESULTS_DIR)
print("\nFiles in dataset directory:")
print(os.listdir(DATA_DIR))

Mounted at /content/drive
Project directory exists: True
Dataset directory exists: True
Results directory: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment_irt

Files in dataset directory:
['skill_builder_data.csv', 'assistments_cleaned.csv', 'assistments_sakt_ready.csv', 'validation_sequences.pkl', 'test_sequences.pkl', 'train_sequences.pkl', 'student_ability_quartiles.csv', 'irt_1pl_output', 'reliability_experiment', 'student_irt_ability_groups.csv', 'reliability_experiment_irt']


## Load the datasets

In [ ]:
# Load ASSISTments 2009 using Latin-1 encoding
DATA_PATH = os.path.join(DATA_DIR, "skill_builder_data.csv")
IRT_PATH = os.path.join(DATA_DIR, "student_irt_ability_groups.csv")

assistments_raw = pd.read_csv(DATA_PATH, encoding="latin1")
irt_groups = pd.read_csv(IRT_PATH)

print("Raw dataset shape:", assistments_raw.shape)
print("Raw dataset columns:")
print(assistments_raw.columns.tolist())

print("\nIRT groups shape:", irt_groups.shape)
print("IRT group columns:")
print(irt_groups.columns.tolist())

print("\nRaw dataset preview:")
display(assistments_raw.head())

print("\nIRT groups preview:")
display(irt_groups.head())

/tmp/ipykernel_2102/1573576510.py:5: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  assistments_raw = pd.read_csv(DATA_PATH, encoding="latin1")


Raw dataset shape: (525534, 30)
Raw dataset columns:
['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']

IRT groups shape: (4163, 4)
IRT group columns:
['user_id', 'theta', 'theta_quartile', 'theta_sd_group']

Raw dataset preview:


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0



IRT groups preview:


,user_id,theta,theta_quartile,theta_sd_group
0,14,-1.597221,Q1,-1.0 to -0.5 SD
1,21825,0.165185,Q3,0 to +0.5 SD
2,51950,0.953936,Q4,+0.5 to +1.0 SD
3,52613,-1.125967,Q2,-1.0 to -0.5 SD
4,53167,-0.213702,Q3,-0.5 to 0 SD


## Make sure that every student has IRT estimates

In [ ]:
# Check data quality and IRT coverage

required_cols = ["user_id", "skill_id", "correct", "order_id"]

print("Missing values in modeling columns:")
print(assistments_raw[required_cols].isna().sum())

print("\nResponse label distribution:")
print(assistments_raw["correct"].value_counts(dropna=False))

print("\nUnique students:", assistments_raw["user_id"].nunique())
print("Unique skills:", assistments_raw["skill_id"].nunique())
print("Unique order IDs:", assistments_raw["order_id"].nunique())

raw_students = set(assistments_raw["user_id"].dropna().unique())
irt_students = set(irt_groups["user_id"].dropna().unique())

print("\nStudents in raw data:", len(raw_students))
print("Students with IRT estimates:", len(irt_students))
print("Students in both:", len(raw_students & irt_students))
print("Raw students without IRT estimates:", len(raw_students - irt_students))

print("\nIRT quartile counts:")
print(irt_groups["theta_quartile"].value_counts().sort_index())

print("\nIRT standard-deviation group counts:")
print(irt_groups["theta_sd_group"].value_counts().sort_index())

Missing values in modeling columns:
user_id         0
skill_id    66326
correct         0
order_id        0
dtype: int64

Response label distribution:
correct
1    357085
0    168449
Name: count, dtype: int64

Unique students: 4217
Unique skills: 123
Unique order IDs: 346860

Students in raw data: 4217
Students with IRT estimates: 4163
Students in both: 4163
Raw students without IRT estimates: 54

IRT quartile counts:
theta_quartile
Q1    1041
Q2    1041
Q3    1040
Q4    1041
Name: count, dtype: int64

IRT standard-deviation group counts:
theta_sd_group
+0.5 to +1.0 SD     381
+1.0 to +1.5 SD     150
-0.5 to 0 SD       1129
-1.0 to -0.5 SD     704
-1.5 to -1.0 SD     299
0 to +0.5 SD        933
< -1.5 SD           351
> +1.5 SD           216
Name: count, dtype: int64


## Clean the data and attach IRT groups

In [ ]:
# Clean ASSISTments 2009 and attach IRT ability groups

model_cols = ["user_id", "skill_id", "correct", "order_id"]

assistments_clean = assistments_raw[model_cols].copy()

# Remove rows missing required modeling fields.
assistments_clean = assistments_clean.dropna(subset=model_cols)

# Keep students with IRT ability estimates.
assistments_clean = assistments_clean.merge(
    irt_groups[["user_id", "theta", "theta_quartile", "theta_sd_group"]],
    on="user_id",
    how="inner",
    validate="many_to_one"
)

# Ensure response and identifier types are consistent.
assistments_clean["user_id"] = assistments_clean["user_id"].astype(int)
assistments_clean["skill_id"] = assistments_clean["skill_id"].astype(int)
assistments_clean["correct"] = assistments_clean["correct"].astype(int)
assistments_clean["order_id"] = assistments_clean["order_id"].astype(int)

# Sort chronologically within each student.
assistments_clean = assistments_clean.sort_values(
    ["user_id", "order_id"],
    kind="mergesort"
).reset_index(drop=True)

print("Cleaned interactions:", len(assistments_clean))
print("Unique students:", assistments_clean["user_id"].nunique())
print("Unique skills:", assistments_clean["skill_id"].nunique())
print("Missing values:")
print(
    assistments_clean[
        ["user_id", "skill_id", "correct", "order_id",
         "theta", "theta_quartile", "theta_sd_group"]
    ].isna().sum()
)

print("\nStudents by IRT quartile:")
print(
    assistments_clean.groupby("theta_quartile")["user_id"]
    .nunique().sort_index()
)

print("\nCorrect-response rate:", round(assistments_clean["correct"].mean(), 6))
display(assistments_clean.head())

Cleaned interactions: 459208
Unique students: 4163
Unique skills: 123
Missing values:
user_id           0
skill_id          0
correct           0
order_id          0
theta             0
theta_quartile    0
theta_sd_group    0
dtype: int64

Students by IRT quartile:
theta_quartile
Q1    1041
Q2    1041
Q3    1040
Q4    1041
Name: user_id, dtype: int64

Correct-response rate: 0.690373


,user_id,skill_id,correct,order_id,theta,theta_quartile,theta_sd_group
0,14,2,0,21617623,-1.597221,Q1,-1.0 to -0.5 SD
1,14,37,0,21617623,-1.597221,Q1,-1.0 to -0.5 SD
2,14,70,0,21617623,-1.597221,Q1,-1.0 to -0.5 SD
3,14,2,1,21617632,-1.597221,Q1,-1.0 to -0.5 SD
4,14,37,1,21617632,-1.597221,Q1,-1.0 to -0.5 SD


## Create the train-test split

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 43
TEST_SIZE = 0.20

# Split student IDs within each IRT quartile.
train_ids = []
test_ids = []

student_abilities = (
    irt_groups[irt_groups["user_id"].isin(assistments_clean["user_id"].unique())]
    [["user_id", "theta_quartile"]]
    .drop_duplicates()
)

for quartile in ["Q1", "Q2", "Q3", "Q4"]:
    quartile_ids = student_abilities.loc[
        student_abilities["theta_quartile"] == quartile, "user_id"
    ].to_numpy()

    quartile_train, quartile_test = train_test_split(
        quartile_ids,
        test_size=TEST_SIZE,
        random_state=SEED,
        shuffle=True
    )

    train_ids.extend(quartile_train)
    test_ids.extend(quartile_test)

train_ids = set(train_ids)
test_ids = set(test_ids)

assistments_train = assistments_clean[
    assistments_clean["user_id"].isin(train_ids)
].copy().reset_index(drop=True)

assistments_test = assistments_clean[
    assistments_clean["user_id"].isin(test_ids)
].copy().reset_index(drop=True)

print("Training interactions:", len(assistments_train))
print("Test interactions:", len(assistments_test))
print("Training students:", len(train_ids))
print("Test students:", len(test_ids))
print("Train/test student overlap:", len(train_ids & test_ids))

print("\nTraining students by quartile:")
print(
    student_abilities[student_abilities["user_id"].isin(train_ids)]
    ["theta_quartile"].value_counts().sort_index()
)

print("\nTest students by quartile:")
print(
    student_abilities[student_abilities["user_id"].isin(test_ids)]
    ["theta_quartile"].value_counts().sort_index()
)

print("\nTraining correct rate:", round(assistments_train["correct"].mean(), 6))
print("Test correct rate:", round(assistments_test["correct"].mean(), 6))

Training interactions: 380092
Test interactions: 79116
Training students: 3328
Test students: 835
Train/test student overlap: 0

Training students by quartile:
theta_quartile
Q1    832
Q2    832
Q3    832
Q4    832
Name: count, dtype: int64

Test students by quartile:
theta_quartile
Q1    209
Q2    209
Q3    208
Q4    209
Name: count, dtype: int64

Training correct rate: 0.691567
Test correct rate: 0.68464


In [ ]:
# Inspect repeated order IDs within each student

def inspect_order_ids(df, name):
    rows_per_event = df.groupby(
        ["user_id", "order_id"], sort=False
    ).size()

    print(f"\n{name}")
    print("Interaction rows:", len(df))
    print("Unique student-order events:", len(rows_per_event))
    print("Events with multiple skill rows:", (rows_per_event > 1).sum())
    print("Maximum rows per event:", rows_per_event.max())
    print("Rows belonging to multi-skill events:",
          rows_per_event[rows_per_event > 1].sum())

inspect_order_ids(assistments_train, "TRAINING")
inspect_order_ids(assistments_test, "TEST")


TRAINING
Interaction rows: 380092
Unique student-order events: 228111
Events with multiple skill rows: 40536
Maximum rows per event: 215
Rows belonging to multi-skill events: 192517

TEST
Interaction rows: 79116
Unique student-order events: 54994
Events with multiple skill rows: 9679
Maximum rows per event: 97
Rows belonging to multi-skill events: 33801


## Remove duplicate rows

In [ ]:
# Count exact duplicates in the columns used for DKT
dedup_cols = ["user_id", "order_id", "skill_id", "correct"]

duplicate_mask = assistments_clean.duplicated(
    subset=dedup_cols,
    keep="first"
)

print("Rows before deduplication:", len(assistments_clean))
print("Exact duplicate rows to remove:", int(duplicate_mask.sum()))

assistments_clean = (
    assistments_clean
    .drop_duplicates(subset=dedup_cols, keep="first")
    .sort_values(["user_id", "order_id", "skill_id"], kind="mergesort")
    .reset_index(drop=True)
)

print("Rows after deduplication:", len(assistments_clean))
print("Remaining duplicates on modeling columns:",
      int(assistments_clean.duplicated(subset=dedup_cols).sum()))
print("Unique students:", assistments_clean["user_id"].nunique())
print("Unique skills:", assistments_clean["skill_id"].nunique())

Rows before deduplication: 459208
Exact duplicate rows to remove: 121207
Rows after deduplication: 338001
Remaining duplicates on modeling columns: 0
Unique students: 4163
Unique skills: 123


## Check data for questions with multiple skills

In [ ]:
# Check how many distinct skill rows remain per student-order event

event_summary = (
    assistments_clean
    .groupby(["user_id", "order_id"], sort=False)
    .agg(
        rows=("skill_id", "size"),
        unique_skills=("skill_id", "nunique"),
        unique_responses=("correct", "nunique")
    )
)

print("Unique student-order events:", len(event_summary))
print("Events with multiple skill rows:",
      int((event_summary["rows"] > 1).sum()))
print("Events with multiple response labels:",
      int((event_summary["unique_responses"] > 1).sum()))
print("Maximum rows per event:", int(event_summary["rows"].max()))

print("\nLargest remaining events:")
display(
    event_summary
    .sort_values("rows", ascending=False)
    .head(10)
    .reset_index()
)

Unique student-order events: 283105
Events with multiple skill rows: 47037
Events with multiple response labels: 0
Maximum rows per event: 4

Largest remaining events:


,user_id,order_id,rows,unique_skills,unique_responses
0,77956,23188771,4,4,1
1,77956,23188878,4,4,1
2,77956,23188915,4,4,1
3,78877,24306072,4,4,1
4,78877,24306159,4,4,1
5,78877,24306308,4,4,1
6,78877,24306378,4,4,1
7,78877,24306507,4,4,1
8,96286,38198158,4,4,1
9,78877,24306665,4,4,1


## Create validation split

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 43
VAL_RATIO = 0.10

# Split only the original training students.
training_student_ids = np.array(sorted(train_ids))

fit_ids, val_ids = train_test_split(
    training_student_ids,
    test_size=VAL_RATIO,
    random_state=SEED,
    shuffle=True
)

fit_ids = set(fit_ids)
val_ids = set(val_ids)

print("Model-fitting students:", len(fit_ids))
print("Validation students:", len(val_ids))
print("Test students:", len(test_ids))

print("\nOverlap checks:")
print("Fit/validation:", len(fit_ids & val_ids))
print("Fit/test:", len(fit_ids & test_ids))
print("Validation/test:", len(val_ids & test_ids))

Model-fitting students: 2995
Validation students: 333
Test students: 835

Overlap checks:
Fit/validation: 0
Fit/test: 0
Validation/test: 0


## Create dataframes and skill mapping

In [ ]:
# Create interaction dataframes using the corrected student split

fit_df = assistments_clean[
    assistments_clean["user_id"].isin(fit_ids)
].copy()

val_df = assistments_clean[
    assistments_clean["user_id"].isin(val_ids)
].copy()

test_df = assistments_clean[
    assistments_clean["user_id"].isin(test_ids)
].copy()

# Build the skill mapping using model-fitting data only
skill_ids = sorted(fit_df["skill_id"].unique())
skill2idx = {skill_id: idx for idx, skill_id in enumerate(skill_ids)}

num_q = len(skill2idx)

print("Number of skills:", num_q)
print("\nInteraction counts:")
print("Model-fitting:", len(fit_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nStudent counts:")
print("Model-fitting:", fit_df["user_id"].nunique())
print("Validation:", val_df["user_id"].nunique())
print("Test:", test_df["user_id"].nunique())

print("\nUnseen validation skills:",
      len(set(val_df["skill_id"].unique()) - set(skill2idx)))
print("Unseen test skills:",
      len(set(test_df["skill_id"].unique()) - set(skill2idx)))

Number of skills: 123

Interaction counts:
Model-fitting: 252186
Validation: 20018
Test: 65797

Student counts:
Model-fitting: 2995
Validation: 333
Test: 835

Unseen validation skills: 0
Unseen test skills: 0


## Build response sequences

In [ ]:
def build_student_sequences(df, skill2idx):
    q_seqs = []
    r_seqs = []
    student_order = []

    df = df.sort_values(
        ["user_id", "order_id", "skill_id"],
        kind="mergesort"
    )

    for user_id, student_df in df.groupby("user_id", sort=False):
        q_seq = student_df["skill_id"].map(skill2idx).to_numpy(
            dtype=np.int64
        )
        r_seq = student_df["correct"].to_numpy(dtype=np.float32)

        if len(q_seq) >= 2:
            student_order.append(user_id)
            q_seqs.append(q_seq)
            r_seqs.append(r_seq)

    return student_order, q_seqs, r_seqs


fit_students, fit_q_seqs, fit_r_seqs = build_student_sequences(
    fit_df, skill2idx
)

val_students, val_q_seqs, val_r_seqs = build_student_sequences(
    val_df, skill2idx
)

test_students, test_q_seqs, test_r_seqs = build_student_sequences(
    test_df, skill2idx
)

for name, students, q_seqs, r_seqs in [
    ("Model-fitting", fit_students, fit_q_seqs, fit_r_seqs),
    ("Validation", val_students, val_q_seqs, val_r_seqs),
    ("Test", test_students, test_q_seqs, test_r_seqs)
]:
    print(f"\n{name}")
    print("Students with sequences:", len(students))
    print("Total sequence rows:", sum(map(len, q_seqs)))
    print("Mean sequence length:", round(np.mean([len(s) for s in q_seqs]), 2))
    print("Maximum sequence length:", max(map(len, q_seqs)))
    print("Skill IDs in valid range:",
          all(np.all((s >= 0) & (s < num_q)) for s in q_seqs))


Model-fitting
Students with sequences: 2910
Total sequence rows: 252101
Mean sequence length: 86.63
Maximum sequence length: 1295
Skill IDs in valid range: True

Validation
Students with sequences: 323
Total sequence rows: 20008
Mean sequence length: 61.94
Maximum sequence length: 1212
Skill IDs in valid range: True

Test
Students with sequences: 814
Total sequence rows: 65776
Mean sequence length: 80.81
Maximum sequence length: 1151
Skill IDs in valid range: True


## Check the repo for chunking implementation

In [ ]:
import os

REPO_DIR = "/content/knowledge-tracing-collection-pytorch"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git

%cd /content/knowledge-tracing-collection-pytorch

from models.utils import match_seq_len, collate_fn
import inspect

print("match_seq_len signature:")
print(inspect.signature(match_seq_len))

print("\nmatch_seq_len implementation:")
print(inspect.getsource(match_seq_len))

Cloning into 'knowledge-tracing-collection-pytorch'...
remote: Enumerating objects: 488, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 488 (delta 22), reused 4 (delta 0), pack-reused 441 (from 1)
Receiving objects: 100% (488/488), 1.11 MiB | 8.90 MiB/s, done.
Resolving deltas: 100% (252/252), done.
/content/knowledge-tracing-collection-pytorch
match_seq_len signature:
(q_seqs, r_seqs, seq_len, pad_val=-1)

match_seq_len implementation:
def match_seq_len(q_seqs, r_seqs, seq_len, pad_val=-1):
    '''
        Args:
            q_seqs: the question(KC) sequences with the size of \
                [batch_size, some_sequence_length]
            r_seqs: the response sequences with the size of \
                [batch_size, some_sequence_length]

            Note that the "some_sequence_length" is not uniform over \
                the whole batch of q_seqs and r_seqs

            seq_len: the sequence length to match the q_s

## Chunk the sequences

In [ ]:
SEQ_LEN = 100

fit_q_chunks, fit_r_chunks = match_seq_len(
    fit_q_seqs, fit_r_seqs, SEQ_LEN
)

val_q_chunks, val_r_chunks = match_seq_len(
    val_q_seqs, val_r_seqs, SEQ_LEN
)

test_q_chunks, test_r_chunks = match_seq_len(
    test_q_seqs, test_r_seqs, SEQ_LEN
)

for name, q_chunks, r_chunks in [
    ("Model-fitting", fit_q_chunks, fit_r_chunks),
    ("Validation", val_q_chunks, val_r_chunks),
    ("Test", test_q_chunks, test_r_chunks),
]:
    lengths = [len(seq) for seq in q_chunks]

    print(f"\n{name}")
    print("Number of chunks:", len(q_chunks))
    print("Chunk lengths:", sorted(set(lengths)))
    print("All q/r lengths match:",
          all(len(q) == len(r) for q, r in zip(q_chunks, r_chunks)))
    print("All chunks have expected length:",
          all(len(q) == SEQ_LEN + 1 for q in q_chunks))
    print("Valid interaction count:",
          sum(int(np.sum((q >= 0) & (r >= 0)))
              for q, r in zip(q_chunks, r_chunks)))
    print("Padding count:",
          sum(int(np.sum((q == -1) | (r == -1)))
              for q, r in zip(q_chunks, r_chunks)))


Model-fitting
Number of chunks: 4544
Chunk lengths: [101]
All q/r lengths match: True
All chunks have expected length: True
Valid interaction count: 252101
Padding count: 206843

Validation
Number of chunks: 427
Chunk lengths: [101]
All q/r lengths match: True
All chunks have expected length: True
Valid interaction count: 20008
Padding count: 23119

Test
Number of chunks: 1231
Chunk lengths: [101]
All q/r lengths match: True
All chunks have expected length: True
Valid interaction count: 65776
Padding count: 58555


In [ ]:
import inspect
from models.dkt import DKT
from models.utils import collate_fn

print("DKT constructor:")
print(inspect.signature(DKT))

print("\nDKT forward method:")
print(inspect.signature(DKT.forward))

print("\nDKT forward implementation:")
print(inspect.getsource(DKT.forward))

print("\nCollate function signature:")
print(inspect.signature(collate_fn))

print("\nCollate function implementation:")
print(inspect.getsource(collate_fn))

DKT constructor:
(num_q, emb_size, hidden_size)

DKT forward method:
(self, q, r)

DKT forward implementation:
    def forward(self, q, r):
        '''
            Args:
                q: the question(KC) sequence with the size of [batch_size, n]
                r: the response sequence with the size of [batch_size, n]

            Returns:
                y: the knowledge level about the all questions(KCs)
        '''
        x = q + self.num_q * r

        h, _ = self.lstm_layer(self.interaction_emb(x))
        y = self.out_layer(h)
        y = self.dropout_layer(y)
        y = torch.sigmoid(y)

        return y


Collate function signature:
(batch, pad_val=-1)

Collate function implementation:
def collate_fn(batch, pad_val=-1):
    '''
        The collate function for torch.utils.data.DataLoader

        Returns:
            q_seqs: the question(KC) sequences with the size of \
                [batch_size, maximum_sequence_length_in_the_batch]
            r_seqs: the response seque

## Create datasets and dataloaders

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128

fit_loader = DataLoader(
    fit_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

batch = next(iter(fit_loader))
q, r, qshft, rshft, mask = batch

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Batch shapes:", q.shape, r.shape, qshft.shape, rshft.shape, mask.shape)
print("Valid targets:", int(mask.sum().item()))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Batch shapes: torch.Size([128, 100]) torch.Size([128, 100]) torch.Size([128, 100]) torch.Size([128, 100]) torch.Size([128, 100])
Valid targets: 10336


/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))


## Define training and eval functions

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from models.dkt import DKT

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def run_epoch(model, loader, optimizer=None):
    """
    If optimizer is supplied, train for one epoch.
    Otherwise, evaluate without updating model parameters.
    """
    training = optimizer is not None
    model.train() if training else model.eval()

    criterion = nn.BCELoss(reduction="none")
    total_loss = 0.0
    total_targets = 0
    all_probs = []
    all_labels = []

    for q, r, qshft, rshft, mask in loader:
        q = q.to(device).long()
        r = r.to(device).long()
        qshft = qshft.to(device).long()
        rshft = rshft.to(device).float()
        mask = mask.to(device).bool()

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            # DKT predicts knowledge levels for every skill at each step.
            y = model(q, r)

            # Select the probability for the next skill at each timestep.
            next_skill_probs = y.gather(
                dim=2,
                index=qshft.unsqueeze(-1)
            ).squeeze(-1)

            # Calculate loss only where the current and next skills are valid.
            valid_probs = next_skill_probs[mask]
            valid_labels = rshft[mask]

            loss = criterion(valid_probs, valid_labels).mean()

            if training:
                loss.backward()
                optimizer.step()

        n_valid = int(mask.sum().item())
        total_loss += loss.item() * n_valid
        total_targets += n_valid

        all_probs.extend(valid_probs.detach().cpu().numpy().tolist())
        all_labels.extend(valid_labels.detach().cpu().numpy().tolist())

    if total_targets == 0:
        return {"loss": float("nan"), "auc": float("nan"),
                "brier": float("nan"), "targets": 0}

    all_probs = np.asarray(all_probs, dtype=np.float64)
    all_labels = np.asarray(all_labels, dtype=np.float64)

    auc = (
        roc_auc_score(all_labels, all_probs)
        if np.unique(all_labels).size == 2
        else float("nan")
    )

    return {
        "loss": total_loss / total_targets,
        "auc": auc,
        "brier": float(np.mean((all_probs - all_labels) ** 2)),
        "targets": total_targets
    }


# Initialize the base DKT architecture.
EMB_SIZE = 100
HIDDEN_SIZE = 100
LEARNING_RATE = 0.001

model = DKT(
    num_q=num_q,
    emb_size=EMB_SIZE,
    hidden_size=HIDDEN_SIZE
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("\nModel initialized")
print("Number of skills:", num_q)
print("Embedding size:", EMB_SIZE)
print("Hidden size:", HIDDEN_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Device: cuda

Model initialized
Number of skills: 123
Embedding size: 100
Hidden size: 100
Learning rate: 0.001
Trainable parameters: 117823


## Fit the model

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import torch
from models.dkt import DKT

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Training configuration
EMB_SIZE = 100
HIDDEN_SIZE = 100
LEARNING_RATE = 0.001
MAX_EPOCHS = 100

os.makedirs(RESULTS_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(RESULTS_DIR, "dkt_best_irt.pt")
HISTORY_PATH = os.path.join(RESULTS_DIR, "dkt_training_history.csv")

# Fresh model and optimizer
model = DKT(
    num_q=num_q,
    emb_size=EMB_SIZE,
    hidden_size=HIDDEN_SIZE
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_auc = -float("inf")
best_epoch = None
best_state = None
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = run_epoch(model, fit_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_auc": train_metrics["auc"],
        "train_brier": train_metrics["brier"],
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["auc"],
        "val_brier": val_metrics["brier"],
        "train_targets": train_metrics["targets"],
        "val_targets": val_metrics["targets"],
    }
    history.append(row)

    improved = (
        np.isfinite(val_metrics["auc"])
        and val_metrics["auc"] > best_val_auc
    )

    if improved:
        best_val_auc = val_metrics["auc"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": best_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auc": best_val_auc,
            "num_q": num_q,
            "emb_size": EMB_SIZE,
            "hidden_size": HIDDEN_SIZE,
            "seed": SEED,
        }, CHECKPOINT_PATH)

    print(
        f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
        f"Train loss {train_metrics['loss']:.5f} | "
        f"Train AUC {train_metrics['auc']:.5f} | "
        f"Train Brier {train_metrics['brier']:.5f} | "
        f"Val loss {val_metrics['loss']:.5f} | "
        f"Val AUC {val_metrics['auc']:.5f} | "
        f"Val Brier {val_metrics['brier']:.5f}"
        + ("  [best checkpoint]" if improved else "")
    )

    # Save progress after every epoch.
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

if best_state is None:
    raise RuntimeError("No valid validation AUC was obtained.")

# Restore the best-validation-AUC checkpoint for subsequent testing.
model.load_state_dict(best_state)

print("\nAll 100 epochs completed.")
print("Best epoch:", best_epoch)
print("Best validation AUC:", round(best_val_auc, 6))
print("Checkpoint:", CHECKPOINT_PATH)
print("Training history:", HISTORY_PATH)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Epoch 001/100 | Train loss 0.64417 | Train AUC 0.61333 | Train Brier 0.22712 | Val loss 0.58374 | Val AUC 0.76456 | Val Brier 0.19791  [best checkpoint]
Epoch 002/100 | Train loss 0.60175 | Train AUC 0.66488 | Train Brier 0.20995 | Val loss 0.54538 | Val AUC 0.79566 | Val Brier 0.18185  [best checkpoint]
Epoch 003/100 | Train loss 0.58806 | Train AUC 0.67635 | Train Brier 0.20543 | Val loss 0.53072 | Val AUC 0.80438 | Val Brier 0.17658  [best checkpoint]
Epoch 004/100 | Train loss 0.58183 | Train AUC 0.68249 | Train Brier 0.20347 | Val loss 0.52315 | Val AUC 0.80874 | Val Brier 0.17412  [best checkpoint]
Epoch 005/100 | Train loss 0.57936 | Train AUC 0.68418 | Train Brier 0.20270 | Val loss 0.51855 | Val AUC 0.81333 | Val Brier 0.17266  [best checkpoint]
Epoch 006/100 | Train loss 0.57667 | Train AUC 0.68826 | Train Brier 0.20175 | Val loss 0.51515 | Val AUC 0.81237 | Val Brier 0.17159
Epoch 007/100 | Train loss 0.57441 | Train AUC 0.69022 | Trai

## Evaluate the model on test data

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from models.dkt import DKT

# --------------------------------------------------
# 1. Load the best-validation-AUC checkpoint
# --------------------------------------------------
CHECKPOINT_PATH = os.path.join(RESULTS_DIR, "dkt_best_irt.pt")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model = DKT(
    num_q=checkpoint["num_q"],
    emb_size=checkpoint["emb_size"],
    hidden_size=checkpoint["hidden_size"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print("Best validation AUC:", round(checkpoint["best_val_auc"], 6))

# --------------------------------------------------
# 2. Predict on all held-out test sequences
#    Store each valid prediction with its student ID
# --------------------------------------------------
prediction_rows = []

with torch.no_grad():
    for q, r, qshft, rshft, mask in test_loader:
        q = q.to(device).long()
        r = r.to(device).long()
        qshft = qshft.to(device).long()
        rshft = rshft.to(device).float()
        mask = mask.to(device).bool()

        probabilities = model(q, r).gather(
            dim=2,
            index=qshft.unsqueeze(-1)
        ).squeeze(-1)

        valid_positions = torch.nonzero(mask, as_tuple=False)

        for batch_idx, time_idx in valid_positions.cpu().numpy():
            prediction_rows.append({
                "student_id": test_students[
                    int(test_chunk_to_student_idx[len(prediction_rows)])
                ] if False else None,
                "target_skill": int(qshft[batch_idx, time_idx].item()),
                "target": int(rshft[batch_idx, time_idx].item()),
                "probability": float(
                    probabilities[batch_idx, time_idx].item()
                )
            })

# Associate chunks with students by rebuilding the chunk-to-student mapping.
# match_seq_len creates consecutive non-overlapping chunks per student.
test_chunk_students = []
for student_id, q_seq in zip(test_students, test_q_seqs):
    n_chunks = (len(q_seq) + 100) // 101
    test_chunk_students.extend([student_id] * n_chunks)

# Rebuild predictions with explicit chunk-level student attribution.
# (The rows above provide the predictions in loader order; this assignment
# aligns each chunk's valid targets to its corresponding student.)
prediction_rows = []
chunk_idx = 0

with torch.no_grad():
    for q, r, qshft, rshft, mask in test_loader:
        q = q.to(device).long()
        r = r.to(device).long()
        qshft = qshft.to(device).long()
        rshft = rshft.to(device).float()
        mask = mask.to(device).bool()

        probabilities = model(q, r).gather(
            dim=2,
            index=qshft.unsqueeze(-1)
        ).squeeze(-1)

        for i in range(q.shape[0]):
            student_id = test_chunk_students[chunk_idx]
            chunk_idx += 1

            valid = mask[i]
            for t in torch.where(valid)[0].cpu().tolist():
                prediction_rows.append({
                    "user_id": student_id,
                    "target": int(rshft[i, t].item()),
                    "probability": float(probabilities[i, t].item())
                })

predictions = pd.DataFrame(prediction_rows)

# --------------------------------------------------
# 3. Attach IRT quartile and SD/uncertainty groups
# --------------------------------------------------
ability_info = (
    assistments_clean[
        ["user_id", "theta_quartile", "theta_sd_group"]
    ]
    .drop_duplicates(subset=["user_id"])
)

predictions = predictions.merge(
    ability_info,
    on="user_id",
    how="left",
    validate="many_to_one"
)

assert predictions["theta_quartile"].notna().all()
assert predictions["theta_sd_group"].notna().all()

# --------------------------------------------------
# 4. Calculate metrics overall and by group
# --------------------------------------------------
def calculate_metrics(df):
    y_true = df["target"].to_numpy(dtype=int)
    y_prob = df["probability"].to_numpy(dtype=float)

    return {
        "students": df["user_id"].nunique(),
        "targets": len(df),
        "correct_rate": float(y_true.mean()),
        "auc": (
            float(roc_auc_score(y_true, y_prob))
            if np.unique(y_true).size == 2 else np.nan
        ),
        "brier": float(np.mean((y_prob - y_true) ** 2))
    }

results = []

overall = calculate_metrics(predictions)
results.append({"group_type": "Overall", "group": "All test students", **overall})

for group_type, column in [
    ("IRT quartile", "theta_quartile"),
    ("IRT SD group", "theta_sd_group")
]:
    for group_name, group_df in predictions.groupby(column, observed=True):
        results.append({
            "group_type": group_type,
            "group": str(group_name),
            **calculate_metrics(group_df)
        })

results_df = pd.DataFrame(results)

# --------------------------------------------------
# 5. Save results and predictions
# --------------------------------------------------
results_path = os.path.join(
    RESULTS_DIR, "dkt_test_results_by_irt_group.csv"
)
predictions_path = os.path.join(
    RESULTS_DIR, "dkt_test_predictions_by_student.csv"
)

results_df.to_csv(results_path, index=False)
predictions.to_csv(predictions_path, index=False)

print("\nHeld-out test results")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\nSaved group results:", results_path)
print("Saved predictions:", predictions_path)
print("Total test predictions:", len(predictions))
print("Unique test students:", predictions["user_id"].nunique())

Loaded checkpoint from epoch: 13
Best validation AUC: 0.819373

Held-out test results
  group_type             group  students  targets  correct_rate      auc    brier
     Overall All test students       814    64545      0.662731 0.823762 0.163089
IRT quartile                Q1       197     8761      0.338318 0.782469 0.189360
IRT quartile                Q2       209    15060      0.587583 0.775495 0.186617
IRT quartile                Q3       208    27465      0.723139 0.796458 0.158256
IRT quartile                Q4       200    13259      0.837318 0.798064 0.129019
IRT SD group   +0.5 to +1.0 SD        70     3747      0.857753 0.796305 0.123182
IRT SD group   +1.0 to +1.5 SD        30      470      0.951064 0.625620 0.100223
IRT SD group      -0.5 to 0 SD       231    19603      0.638984 0.776965 0.179864
IRT SD group   -1.0 to -0.5 SD       136     9348      0.456354 0.761536 0.198776
IRT SD group   -1.5 to -1.0 SD        50     1599      0.317699 0.741327 0.196225
IRT SD group

### DKT Test Results by IRT Ability Group

**Best checkpoint:** Epoch 13  
**Best validation AUC:** 0.819373

#### Overall and IRT Quartile Results

| Group Type | Group | Students | Targets | Correct Rate | AUC | Brier Score |
|---|---|---:|---:|---:|---:|---:|
| Overall | All test students | 814 | 64,545 | 0.662731 | 0.823762 | 0.163089 |
| IRT quartile | Q1 | 197 | 8,761 | 0.338318 | 0.782469 | 0.189360 |
| IRT quartile | Q2 | 209 | 15,060 | 0.587583 | 0.775495 | 0.186617 |
| IRT quartile | Q3 | 208 | 27,465 | 0.723139 | 0.796458 | 0.158256 |
| IRT quartile | Q4 | 200 | 13,259 | 0.837318 | 0.798064 | 0.129019 |

#### IRT Ability SD Group Results

| Group Type | Group | Students | Targets | Correct Rate | AUC | Brier Score |
|---|---|---:|---:|---:|---:|---:|
| IRT SD group | < -1.5 SD | 68 | 1,554 | 0.077864 | 0.791122 | 0.134065 |
| IRT SD group | -1.5 to -1.0 SD | 50 | 1,599 | 0.317699 | 0.741327 | 0.196225 |
| IRT SD group | -1.0 to -0.5 SD | 136 | 9,348 | 0.456354 | 0.761536 | 0.198776 |
| IRT SD group | -0.5 to 0 SD | 231 | 19,603 | 0.638984 | 0.776965 | 0.179864 |
| IRT SD group | 0 to +0.5 SD | 190 | 27,858 | 0.765669 | 0.803977 | 0.146525 |
| IRT SD group | +0.5 to +1.0 SD | 70 | 3,747 | 0.857753 | 0.796305 | 0.123182 |
| IRT SD group | +1.0 to +1.5 SD | 30 | 470 | 0.951064 | 0.625620 | 0.100223 |
| IRT SD group | > +1.5 SD | 39 | 366 | 0.994536 | 0.826923 | 0.081682 |